**Innovation Management · A.Y. 2026/27 · Politecnico di Torino** · Project Work · **Team 28 · Flying drones · client: Leonardo**

| | |
|---|---|
| **Question (Step 1, Q4)** | Is the field patent-heavy, or does it rely on secrecy? |
| **Approach** | Compare flying drones with similar fields, and split drones into military, dual-use (surveillance) and civil uses. |
| **Based on** | The counting rules of `tutorial_0/02_step1_general_description.ipynb`: family level, earliest priority year, patents of invention only, last incomplete years excluded. |
| **PATSTAT environment** | TEST while developing, then `ENV = "PROD"` and Run All for the final numbers. |
| **Outputs** | `outputs/q4_secrecy/`: CSV tables and PNG figures for the deck. |

## Why this question is tricky for drones

Patents are **public**: an applicant trades secrecy for 20 years of protection. A field that relies on secrecy shows up in PATSTAT as *missing* patents, so we cannot count secrecy directly. We look at indirect signals instead:

| Signal | Patent-heavy field | Secrecy-reliant field |
|---|---|---|
| Volume and growth of families | many, growing | few, flat, relative to the size of the industry |
| International protection (family size, PCT, ≥ 2 IP5 offices) | inventions filed in many offices | inventions filed in one office only (a secrecy order usually also restricts foreign filing while it is in force, with some exceptions) |
| Who files (applicant sector) | firms, universities | few government / defence applicants |
| Military vs civil drones | military share in line with the market | military share far below the weight of defence in the drone market |
| Large defence primes (Leonardo, Lockheed, ...) | file many drone patents | file few, compared with their R&D size |

Defence is the extreme case: in Italy, the US, the UK and France, an application touching national defence can be placed under a **secrecy order**: it is not published while the order is in force (orders can be renewed for years, or lifted), so it is missing from PATSTAT or appears only much later (Prof. Ascione suggested looking at **dual-use** technology for this reason). None of the indicators below measures secrecy directly, and PATSTAT has no denominator such as market size or R&D spending: the results are **consistent with** or **not consistent with** secrecy, never a proof. The comparison with civil drones and with similar civil fields is what makes the argument.

## 1. Setup and configuration

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from matplotlib.ticker import PercentFormatter
from sqlalchemy import and_, case, distinct, func, or_

from epo.tipdata.patstat import PatstatClient
from epo.tipdata.patstat.database.models import (
    TLS201_APPLN,
    TLS206_PERSON,
    TLS207_PERS_APPLN,
    TLS224_APPLN_CPC,
)

In [ ]:
# ---------------------------------------------------------------- environment
ENV = "TEST"                       # develop on TEST, switch to "PROD" for the final numbers
OUTPUT_DIR = Path("outputs/q4_secrecy")

# ---------------------------------------------------------------- time window (same rules as Step 1)
YEAR_MIN, YEAR_MAX = 2000, 2025
TRUNCATED_YEARS = 3                # last priority years still incomplete in PATSTAT
LAST_COMPLETE_YEAR = YEAR_MAX - TRUNCATED_YEARS
IPR_TYPE = "PI"                    # patents of invention only
IP5_OFFICES = ("EP", "US", "CN", "JP", "KR")
TOP_N = 15

# ---------------------------------------------------------------- fields (CPC prefixes, blanks are ignored)
DRONE_CPC = ("B64U", "B64C39/024", "B64C2201")          # same definition as the Step 1 notebook
MILITARY_CPC = (                                        # military uses of drones
    "B64U2101/15",   # UAVs for conventional or electronic warfare
    "B64U2101/16",   # . for controlling, capturing or immobilising other vehicles
    "B64U2101/17",   # . for detecting, disrupting or countering communications
    "B64U2101/18",   # . for dropping bombs; for firing ammunition
    "B64U2101/19",   # . for use as targets or decoys
    "F41",           # weapons (co-classification)
    "F42B10",        # projectiles or missiles characterised by aerodynamic features
    "F42B12",        # projectiles or missiles characterised by the warhead
    "F42B15",        # self-propelled projectiles or missiles
    # not the whole of F42: F42B 4/00 is fireworks
)
DUAL_USE_CPC = ("B64U2101/31",)                          # UAVs for surveillance: civil and military
BENCHMARKS = {                                          # similar fields, for comparison
    "Manned aircraft": ("B64C", "B64D"),               # drone families are removed below
    "Autonomous road vehicles": ("B60W60",),
}

# ---------------------------------------------------------------- defence and aerospace primes (applicant names, PATSTAT psn_name)
PRIMES = {
    "Leonardo (incl. Finmeccanica, Selex, Alenia, AgustaWestland)":
        ("%LEONARDO S%P%A%", "%LEONARDO SPA%", "%FINMECCANICA%", "%SELEX%", "%ALENIA%", "%AGUSTAWESTLAND%"),
    "Lockheed Martin": ("%LOCKHEED%",),
    "Northrop Grumman": ("%NORTHROP%",),
    "Raytheon / RTX": ("%RAYTHEON%",),
    "Boeing": ("%BOEING%",),
    "Airbus": ("%AIRBUS%",),
    "BAE Systems": ("%BAE SYSTEMS%",),
    "Thales": ("%THALES%",),
    "General Atomics": ("%GENERAL ATOMICS%",),
    "Elbit Systems": ("%ELBIT%",),
    "Israel Aerospace Industries": ("%ISRAEL AEROSPACE%",),
    "AeroVironment": ("%AEROVIRONMENT%",),
    "DJI (civil benchmark)": ("%SZ DJI%", "%DJI TECHNOLOGY%", "%DA-JIANG%", "%DA JIANG%"),
}

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
patstat = PatstatClient(env=ENV)
db = patstat.orm()
print(f"env: {ENV} | priority years {YEAR_MIN}-{LAST_COMPLETE_YEAR} (complete years only)")

## 2. Defining the groups

Every group is a query of `docdb_family_id`s, selected by CPC prefix as in Step 1. Drone families are split into three non-overlapping groups:

- **military**: drone families with a warfare code (B64U 2101/15–19) or co-classified in weapons (F41) or missiles and projectiles (F42B 10, 12, 15);
- **dual-use (surveillance)**: drone families with B64U 2101/31, not military;
- **civil and other**: all remaining drone families.

The benchmarks exclude every drone family, so that they do not overlap with the sector.

In [ ]:
def symbol_prefix_condition(symbol_column, codes):
    """OR-condition: the blank-stripped classification symbol starts with one of the codes."""
    symbol = func.replace(symbol_column, " ", "")
    return or_(*(symbol.like(code.replace(" ", "") + "%") for code in codes))


def cpc_family_ids(prefixes):
    """Families with at least one CPC symbol starting with one of the prefixes."""
    return (
        db.query(TLS201_APPLN.docdb_family_id.label("docdb_family_id"))
        .join(TLS224_APPLN_CPC, TLS201_APPLN.appln_id == TLS224_APPLN_CPC.appln_id)
        .filter(symbol_prefix_condition(TLS224_APPLN_CPC.cpc_class_symbol, prefixes))
        .distinct()
    )


def subset(ids, keep=None, drop=()):
    """Families of ids that are also in keep (if given) and in none of drop."""
    q = ids
    if keep is not None:
        q = q.filter(TLS201_APPLN.docdb_family_id.in_(keep.subquery().select()))
    for other in drop:
        q = q.filter(~TLS201_APPLN.docdb_family_id.in_(other.subquery().select()))
    return q


drones = cpc_family_ids(DRONE_CPC)
military = subset(cpc_family_ids(MILITARY_CPC), keep=drones)
dual_use = subset(cpc_family_ids(DUAL_USE_CPC), keep=drones, drop=[military])
civil = subset(drones, drop=[military, dual_use])

GROUPS = {
    "Flying drones (all)": drones,
    "  military": military,
    "  dual-use (surveillance)": dual_use,
    "  civil and other": civil,
}
GROUPS.update({name: subset(cpc_family_ids(codes), drop=[drones]) for name, codes in BENCHMARKS.items()})
DRONE_SPLIT = ["  military", "  dual-use (surveillance)", "  civil and other"]

### 2.1 Precision check of the military group

Random titles of military drone families: read them and remove codes that bring in non-military inventions. Mention the result in the appendix.

In [ ]:
from epo.tipdata.patstat.database.models import TLS202_APPLN_TITLE

sample_military = patstat.df(
    db.query(TLS201_APPLN.docdb_family_id, func.min(TLS202_APPLN_TITLE.appln_title).label("title"))
    .join(TLS202_APPLN_TITLE, TLS201_APPLN.appln_id == TLS202_APPLN_TITLE.appln_id)
    .filter(TLS201_APPLN.docdb_family_id.in_(military.subquery().select()))
    .group_by(TLS201_APPLN.docdb_family_id)
    .order_by(func.rand())
    .limit(25)
)
sample_military

## 3. Family profile

One row per family, computed inside the database: priority year, family size, number of offices (excluding `WO`), number of IP5 offices, PCT, granted. Only complete priority years are kept.

In [ ]:
IS_PCT_APPLN = and_(TLS201_APPLN.appln_auth == "WO", TLS201_APPLN.appln_kind.like("W%"))


def family_profile(ids):
    a = TLS201_APPLN
    year = func.min(a.earliest_filing_year)
    return (
        db.query(
            a.docdb_family_id.label("fid"),
            year.label("year"),
            func.max(a.docdb_family_size).label("family_size"),
            func.count(distinct(case((a.appln_auth != "WO", a.appln_auth)))).label("n_offices"),   # WO = PCT, not an office of protection
            func.count(distinct(case((a.appln_auth.in_(IP5_OFFICES), a.appln_auth)))).label("n_ip5"),
            func.max(case((IS_PCT_APPLN, 1), else_=0)).label("is_pct"),
            func.max(case((a.granted == "Y", 1), else_=0)).label("is_granted"),
        )
        .filter(a.ipr_type == IPR_TYPE, a.docdb_family_id.in_(ids.subquery().select()))
        .group_by(a.docdb_family_id)
        .having(year.between(YEAR_MIN, LAST_COMPLETE_YEAR))
        .subquery()
    )


PROFILES = {name: family_profile(ids) for name, ids in GROUPS.items()}

## 4. Indicators of patenting intensity

- **families**: number of inventions patented;
- **mean family size**: applications per invention, i.e. effort to protect it;
- **single office**: share of inventions filed at one patent office only, not counting PCT (`WO`) applications, which are not a protection in themselves (an `EP` application can still cover several countries);
- **≥ 2 IP5 offices**: share protected in at least two of the five largest offices;
- **via PCT** and **granted (any office)**.

In [ ]:
def summary(name, p):
    q = db.query(
        func.count().label("families"),
        func.avg(p.c.family_size * 1.0).label("mean family size"),
        func.avg(case((p.c.n_offices <= 1, 1.0), else_=0.0)).label("single office"),   # 0 = PCT only, not yet national
        func.avg(case((p.c.n_ip5 >= 2, 1.0), else_=0.0)).label(">= 2 IP5 offices"),
        func.avg(p.c.is_pct * 1.0).label("via PCT"),
        func.avg(p.c.is_granted * 1.0).label("granted"),
    )
    return patstat.df(q).assign(group=name)


indicators = pd.concat([summary(n, p) for n, p in PROFILES.items()], ignore_index=True).set_index("group")
SHARE_COLS = ["single office", ">= 2 IP5 offices", "via PCT", "granted"]
indicators[SHARE_COLS] = indicators[SHARE_COLS] * 100
indicators["share of drone families (%)"] = (
    indicators["families"] / indicators.loc["Flying drones (all)", "families"] * 100
).where(indicators.index.isin(DRONE_SPLIT))
indicators.style.format({"families": "{:,.0f}", "mean family size": "{:.2f}",
                         "share of drone families (%)": "{:.1f}%", **{c: "{:.1f}%" for c in SHARE_COLS}},
                        na_rep="")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5), sharey=True)
for ax, col in zip(axes, ["single office", ">= 2 IP5 offices", "via PCT"]):
    ax.barh(indicators.index, indicators[col])
    ax.set(title=col, xlim=(0, 100))
    ax.xaxis.set_major_formatter(PercentFormatter(100))
axes[0].invert_yaxis()
fig.suptitle("How widely are inventions protected? (share of families)")
plt.tight_layout()
fig.savefig(OUTPUT_DIR / f"q4_protection_scope_{ENV}.png", dpi=200, bbox_inches="tight")
plt.show()

## 5. Growth: drones vs benchmarks, and military vs civil drones

Families per priority year, indexed to the first year with data (= 100) so that fields of different size can be compared.

In [ ]:
def families_per_year(name, p):
    q = db.query(p.c.year, func.count().label("families")).group_by(p.c.year)
    return patstat.df(q).assign(group=name)


per_year = pd.concat([families_per_year(n, p) for n, p in PROFILES.items()], ignore_index=True)
per_year_wide = per_year.pivot_table(index="year", columns="group", values="families", fill_value=0)
per_year_wide = per_year_wide.reindex(index=range(YEAR_MIN, LAST_COMPLETE_YEAR + 1), columns=list(GROUPS), fill_value=0)


def indexed(series):
    nonzero = series[series > 0]
    return series / nonzero.iloc[0] * 100 if len(nonzero) else series * float("nan")


fig, axes = plt.subplots(1, 2, figsize=(15, 4.5))
for name in ["Flying drones (all)", *BENCHMARKS]:
    axes[0].plot(per_year_wide.index, indexed(per_year_wide[name]), marker="o", ms=3, label=name.strip())
axes[0].set(title="Families per year (first year with data = 100)", xlabel="priority year", yscale="log")
axes[0].legend()

split = per_year_wide[DRONE_SPLIT]
split_share = split.div(split.sum(axis=1).replace(0, float("nan")), axis=0) * 100
axes[1].stackplot(split_share.index, split_share.T.values, labels=[c.strip() for c in DRONE_SPLIT], alpha=0.85)
axes[1].set(title="Drone families by use (share of drone families)", xlabel="priority year", ylim=(0, 100))
axes[1].yaxis.set_major_formatter(PercentFormatter(100))
axes[1].legend(loc="upper left")
plt.tight_layout()
fig.savefig(OUTPUT_DIR / f"q4_growth_and_split_{ENV}.png", dpi=200, bbox_inches="tight")
plt.show()

## 6. Who files? Applicant sectors (fractional counts)

PATSTAT classifies applicants in `psn_sector` (COMPANY, UNIVERSITY, GOV NON-PROFIT, INDIVIDUAL, HOSPITAL, UNKNOWN). Each family is split equally among its distinct applicants, so each group sums to 100%. A small government share in military drones would be *consistent with* secrecy (governments and defence agencies tend not to publish), but it is not a proof. Families without any applicant in `tls207` are left out of the mix: their share is reported in the coverage table, mention it if it is large.

In [ ]:
def applicant_sector_mix(name, p):
    sector = func.coalesce(func.nullif(func.trim(TLS206_PERSON.psn_sector), ""), "UNKNOWN")
    pairs = (
        db.query(p.c.fid.label("fid"), TLS207_PERS_APPLN.person_id, sector.label("sector"))
        .join(TLS201_APPLN, TLS201_APPLN.docdb_family_id == p.c.fid)
        .join(TLS207_PERS_APPLN, TLS207_PERS_APPLN.appln_id == TLS201_APPLN.appln_id)
        .join(TLS206_PERSON, TLS206_PERSON.person_id == TLS207_PERS_APPLN.person_id)
        .filter(TLS201_APPLN.ipr_type == IPR_TYPE, TLS207_PERS_APPLN.applt_seq_nr > 0)
        .distinct()
        .subquery()
    )
    k = db.query(pairs.c.fid, func.count().label("k")).group_by(pairs.c.fid).subquery()
    q = (
        db.query(pairs.c.sector, func.sum(1.0 / k.c.k).label("n_frac"))
        .join(k, k.c.fid == pairs.c.fid)
        .group_by(pairs.c.sector)
    )
    return patstat.df(q).assign(group=name)


def families_with_applicant(p):
    """Families of the profile with at least one applicant in tls207 (the rest is missing from the mix)."""
    q = (
        db.query(func.count(distinct(p.c.fid)).label("n"))
        .select_from(p)
        .join(TLS201_APPLN, TLS201_APPLN.docdb_family_id == p.c.fid)
        .join(TLS207_PERS_APPLN, TLS207_PERS_APPLN.appln_id == TLS201_APPLN.appln_id)
        .filter(TLS201_APPLN.ipr_type == IPR_TYPE, TLS207_PERS_APPLN.applt_seq_nr > 0)
    )
    return int(patstat.df(q)["n"].iloc[0])


sectors = pd.concat([applicant_sector_mix(n, p) for n, p in PROFILES.items()], ignore_index=True)
sector_shares = sectors.pivot_table(index="group", columns="sector", values="n_frac", aggfunc="sum", fill_value=0)
sector_shares = (sector_shares.div(sector_shares.sum(axis=1), axis=0) * 100).reindex(list(GROUPS)).dropna(how="all")
sector_shares = sector_shares[sector_shares.sum().sort_values(ascending=False).index]

# Coverage: share of families with applicant data. The mix above is computed on these families only.
applicant_coverage = pd.Series(
    {name: families_with_applicant(p) for name, p in PROFILES.items()}, name="families with applicant"
).to_frame()
applicant_coverage["families"] = indicators["families"]
applicant_coverage["coverage (%)"] = applicant_coverage["families with applicant"] / applicant_coverage["families"] * 100
display(applicant_coverage.style.format({"coverage (%)": "{:.1f}%"}))
sector_shares.style.format("{:.1f}%")

In [ ]:
if sector_shares.empty:
    print("No applicant data for any group: nothing to plot.")
else:
    fig, ax = plt.subplots(figsize=(12, 4))
    sector_shares.plot.barh(stacked=True, ax=ax, width=0.8)
    ax.set(title="Applicant sectors (share of families, fractional counts)", xlim=(0, 100), ylabel="")
    ax.xaxis.set_major_formatter(PercentFormatter(100))
    ax.invert_yaxis()
    ax.legend(loc="center left", bbox_to_anchor=(1, 0.5))
    fig.savefig(OUTPUT_DIR / f"q4_applicant_sectors_{ENV}.png", dpi=200, bbox_inches="tight")
    plt.show()

## 7. Defence primes vs DJI: how many drone families do they file?

Number of drone families (complete years) where the company is among the applicants, matched on the PATSTAT standardised name (`psn_name`). Check the matched names in the second table: patterns can catch unrelated companies, and one company can appear under several names (name harmonisation is Step 2).

In [ ]:
def name_condition(patterns):
    return or_(*(TLS206_PERSON.psn_name.like(p) for p in patterns))


def applicant_families(p, patterns):
    return (
        db.query(p.c.fid.label("fid"), TLS206_PERSON.psn_name.label("psn_name"))
        .join(TLS201_APPLN, TLS201_APPLN.docdb_family_id == p.c.fid)
        .join(TLS207_PERS_APPLN, TLS207_PERS_APPLN.appln_id == TLS201_APPLN.appln_id)
        .join(TLS206_PERSON, TLS206_PERSON.person_id == TLS207_PERS_APPLN.person_id)
        .filter(TLS201_APPLN.ipr_type == IPR_TYPE, TLS207_PERS_APPLN.applt_seq_nr > 0, name_condition(patterns))
        .distinct()
    )


prime_rows, prime_names = [], []
for company, patterns in PRIMES.items():
    row = {"company": company}
    for name in ["Flying drones (all)", "  military"]:
        matches = patstat.df(applicant_families(PROFILES[name], patterns))
        row[name.strip() + " families"] = matches["fid"].nunique()
        if name == "Flying drones (all)":
            prime_names.append(matches.groupby("psn_name")["fid"].nunique().rename("families")
                               .reset_index().assign(company=company))
    prime_rows.append(row)

primes = pd.DataFrame(prime_rows).set_index("company").sort_values("Flying drones (all) families", ascending=False)
prime_names = pd.concat(prime_names, ignore_index=True) if prime_names else pd.DataFrame()
primes

In [ ]:
# Which psn_names were matched? Remove patterns that catch the wrong companies.
prime_names.sort_values(["company", "families"], ascending=[True, False])

## 8. Top applicants in military and in civil drones

In [ ]:
def top_applicants(p, n=TOP_N):
    fam = distinct(p.c.fid)
    q = (
        db.query(TLS206_PERSON.psn_name, TLS206_PERSON.person_ctry_code, TLS206_PERSON.psn_sector,
                 func.count(fam).label("families"))
        .select_from(p)
        .join(TLS201_APPLN, TLS201_APPLN.docdb_family_id == p.c.fid)
        .join(TLS207_PERS_APPLN, TLS207_PERS_APPLN.appln_id == TLS201_APPLN.appln_id)
        .join(TLS206_PERSON, TLS206_PERSON.person_id == TLS207_PERS_APPLN.person_id)
        .filter(TLS201_APPLN.ipr_type == IPR_TYPE, TLS207_PERS_APPLN.applt_seq_nr > 0)
        .group_by(TLS206_PERSON.psn_name, TLS206_PERSON.person_ctry_code, TLS206_PERSON.psn_sector)
        .order_by(func.count(fam).desc())
        .limit(n)
    )
    return patstat.df(q)


top_military = top_applicants(PROFILES["  military"])
top_civil = top_applicants(PROFILES["  civil and other"])
display(top_military)
display(top_civil)

## 9. Export for the deck

In [ ]:
exports = {
    "q4_indicators": indicators,
    "q4_families_per_year": per_year_wide,
    "q4_applicant_sectors": sector_shares,
    "q4_applicant_coverage": applicant_coverage,
    "q4_defence_primes": primes,
    "q4_defence_primes_matched_names": prime_names,
    "q4_top_applicants_military": top_military,
    "q4_top_applicants_civil": top_civil,
    "q4_military_sample_titles": sample_military,
}
for name, table in exports.items():
    table.to_csv(OUTPUT_DIR / f"{name}_{ENV}.csv")
sorted(p.name for p in OUTPUT_DIR.iterdir())

## 10. How to read the results (draft for the slide)

The statements below are **hypotheses to check against the PROD numbers**, not conclusions: keep a statement only if the data support it, and rewrite it otherwise. Use cautious wording ("consistent with", "suggests").

1. **Is the field patent-heavy?** If drone families grew faster than manned aircraft and autonomous road vehicles (`__`× vs `__`× vs `__`×) and a large share is filed in ≥ 2 IP5 offices (`__`% vs `__`%), the civil drone field looks patent-heavy.
2. **Does the military part rely on secrecy?** If the military share is small (`__`% of drone families) compared with the weight of defence in the drone market (cite an external market source, PATSTAT cannot give it), if military families are more often filed in one office only (`__`% vs `__`% civil) and if government applicants are rare (`__`%), this is *consistent with* secrecy orders or trade secrets in defence. Other explanations to mention: B64U 2101 use codes are assigned to few families, defence firms may file under generic codes, and the CPC selection can include false positives (see the precision check in 2.1).
3. **Defence primes vs DJI.** Leonardo `__` drone families, Lockheed `__`, Boeing `__`, against `__` for DJI. Few filings by large primes, relative to their size, suggest (do not prove) a lower propensity to patent.
4. **Implication for Leonardo.** If the points above hold, the visible patent landscape is driven by civil and dual-use players, so patent data are useful to monitor dual-use technologies (surveillance, communications, autonomy), while the defence segment needs other sources (contracts, procurement, publications).

State the limits in the technical appendix: no market or R&D denominator, CPC-based selection, B64U2101 use codes are not assigned to every drone family while co-classification can bring in false positives (so the military share can be under- or over-estimated), secret patents are by definition not observable, `psn_name` matching is approximate.